# Notebook 20: Anomaly Detection in Hetionet

## Purpose
Identify metapaths with biological signal beyond degree structure using z-scores.

**Goal**: z = (observed - expected) / √variance → FDR-corrected p-values → significant anomalies

## Method
1. Load deployment recommendation from notebook 18 (best method)
2. Compute expected pathway counts using best method
3. Load variance estimates from notebook 19
4. Compute observed pathway counts in Hetionet
5. Calculate z-scores for each (source, target) pair
6. Apply FDR correction (Benjamini-Hochberg)
7. Identify and characterize significant anomalies

## Inputs
- results/null_approximation/deployment_recommendation.json
- results/variance_estimates/*.npz and *.pkl
- data/hetionet-v1.0/hetmat/edges/*.sparse.npz

## Outputs
- results/anomaly_detection/all_z_scores.csv
- results/anomaly_detection/significant_FDR05.csv
- results/anomaly_detection/summary_by_metapath.csv
- results/anomaly_detection/plots/*.png

## Dependencies
- Notebook 17 must PASS
- Notebook 18 must complete (deployment recommendation)
- Notebook 19 must complete (variance estimation)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
from scipy.stats import norm, pearsonr
from statsmodels.stats.multitest import multipletests
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

repo_dir = Path.cwd().parent
data_dir = repo_dir / 'data'
results_dir = repo_dir / 'results' / 'anomaly_detection'
results_dir.mkdir(parents=True, exist_ok=True)
(results_dir / 'plots').mkdir(parents=True, exist_ok=True)

print(f"Repository: {repo_dir}")
print(f"Results will be saved to: {results_dir}")

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## Check Prerequisites

In [ ]:
# Check compositional validation
validation_file = repo_dir / 'results' / 'compositional_validation' / 'validation_summary.json'
if not validation_file.exists():
    raise FileNotFoundError("Run notebook 17_compositional_validation.ipynb first!")

with open(validation_file, 'r') as f:
    validation_summary = json.load(f)

if validation_summary['overall_decision'] == 'FAILED':
    raise ValueError("Compositional validation FAILED!")

print("✓ Compositional validation: PASSED")
print(f"  Mean Pearson r: {validation_summary['overall_mean_pearson_r']:.4f}")

# Check deployment recommendation
recommendation_file = repo_dir / 'results' / 'null_approximation' / 'deployment_recommendation.json'
if not recommendation_file.exists():
    raise FileNotFoundError("Run notebook 18_null_approximation_comparison.ipynb first!")

with open(recommendation_file, 'r') as f:
    deployment_rec = json.load(f)

print("\n✓ Deployment recommendation loaded")
print(f"  Method: {deployment_rec['method']}")
print(f"  N permutations: {deployment_rec['N_required']}")
print(f"  Accuracy: r = {deployment_rec['accuracy']:.4f}" if deployment_rec['accuracy'] else "")

# Check variance estimates
variance_dir = repo_dir / 'results' / 'variance_estimates'
if not variance_dir.exists():
    raise FileNotFoundError("Run notebook 19_variance_estimation.ipynb first!")

variance_files = list(variance_dir.glob('*_variance.npz'))
print(f"\n✓ Variance estimates found: {len(variance_files)} metapaths")

print("\n" + "="*70)
print("All prerequisites satisfied!")
print("="*70)

## Configuration

In [ ]:
# Papermill parameters
fdr_threshold = 0.05
min_pathway_count = 1
random_seed = 42

In [ ]:
# Metapaths to analyze
metapaths_2hop = [
    {'name': 'CbGpPW', 'edge1': 'CbG', 'edge2': 'GpPW'},
    {'name': 'CtDaG', 'edge1': 'CtD', 'edge2': 'DaG'},
    {'name': 'CbGaD', 'edge1': 'CbG', 'edge2': 'GaD'},
    {'name': 'CrCbG', 'edge1': 'CrC', 'edge2': 'CbG'},
    {'name': 'CbGiG', 'edge1': 'CbG', 'edge2': 'GiG'},
    {'name': 'CpDaG', 'edge1': 'CpD', 'edge2': 'DaG'},
    {'name': 'CbGpBP', 'edge1': 'CbG', 'edge2': 'GpBP'},
    {'name': 'CbGpCC', 'edge1': 'CbG', 'edge2': 'GpCC'},
]

print(f"Analyzing {len(metapaths_2hop)} metapaths")
print(f"FDR threshold: {fdr_threshold}")
print(f"Minimum pathway count: {min_pathway_count}")
print(f"Method: {deployment_rec['method']}")

## Helper Functions

In [ ]:
def load_edge_matrix(edge_type, base_dir='hetionet-v1.0', perm_id=None):
    """
    Load edge matrix from Hetionet or permutation.
    
    Args:
        edge_type: Edge type abbreviation
        base_dir: Base directory name
        perm_id: Permutation ID (None for Hetionet)
    
    Returns:
        scipy.sparse matrix
    """
    if perm_id is None:
        edge_file = data_dir / base_dir / 'hetmat' / 'edges' / f'{edge_type}.sparse.npz'
    else:
        perm_dir = f'{perm_id:03d}.hetmat'
        edge_file = data_dir / base_dir / 'permutations' / perm_dir / 'edges' / f'{edge_type}.sparse.npz'
    
    if not edge_file.exists():
        raise FileNotFoundError(f"Edge file not found: {edge_file}")
    
    return sp.load_npz(str(edge_file))

def compute_empirical_edge_probs(edge_type, perm_ids):
    """
    Compute empirical edge probabilities from permutations.
    
    Args:
        edge_type: Edge type abbreviation
        perm_ids: List of permutation IDs
    
    Returns:
        scipy.sparse matrix: Edge probabilities
    """
    first_matrix = load_edge_matrix(edge_type, perm_id=perm_ids[0])
    edge_sum = sp.csr_matrix(first_matrix.shape, dtype=np.float64)
    
    for perm_id in perm_ids:
        edge_matrix = load_edge_matrix(edge_type, perm_id=perm_id)
        edge_sum = edge_sum + edge_matrix.astype(np.float64)
    
    return edge_sum / len(perm_ids)

def load_variance_estimates(metapath):
    """
    Load variance matrix and lookup table for a metapath.
    
    Args:
        metapath: Metapath name
    
    Returns:
        tuple: (variance_matrix, variance_lookup_data)
    """
    variance_dir = repo_dir / 'results' / 'variance_estimates'
    
    variance_file = variance_dir / f'{metapath}_variance.npz'
    lookup_file = variance_dir / f'{metapath}_variance_lookup.pkl'
    
    if not variance_file.exists() or not lookup_file.exists():
        raise FileNotFoundError(f"Variance files not found for {metapath}")
    
    variance_matrix = sp.load_npz(str(variance_file))
    
    with open(lookup_file, 'rb') as f:
        lookup_data = pickle.load(f)
    
    return variance_matrix, lookup_data

print("Helper functions defined")

## Compute Expected Pathway Counts

In [ ]:
print("\nComputing expected pathway counts...\n")
print(f"Method: {deployment_rec['method']}")
print(f"N permutations: {deployment_rec['N_required']}\n")

expected_pathways = {}

# Use empirical method (learned formula and ML model use same approach)
if deployment_rec['N_required'] > 0:
    perm_ids = list(range(1, deployment_rec['N_required'] + 1))
    
    for mp_info in metapaths_2hop:
        metapath = mp_info['name']
        edge1 = mp_info['edge1']
        edge2 = mp_info['edge2']
        
        print(f"Computing expected for {metapath}...")
        
        # Compute edge probabilities
        edge1_probs = compute_empirical_edge_probs(edge1, perm_ids)
        edge2_probs = compute_empirical_edge_probs(edge2, perm_ids)
        
        # Compositional calculation
        expected = edge1_probs @ edge2_probs
        
        expected_pathways[metapath] = expected
        print(f"  Shape: {expected.shape}, nnz: {expected.nnz:,}")
else:
    # Analytical formula (0 permutations)
    print("Using analytical formula (0 permutations)...")
    # Note: Would implement actual analytical formula here
    # For now, use simplified degree-based approach
    raise NotImplementedError("Analytical formula not yet implemented")

print(f"\n✓ Expected pathway counts computed for {len(expected_pathways)} metapaths")

## Compute Observed Pathway Counts in Hetionet

In [ ]:
print("\nComputing observed pathway counts in Hetionet...\n")

observed_pathways = {}

for mp_info in metapaths_2hop:
    metapath = mp_info['name']
    edge1 = mp_info['edge1']
    edge2 = mp_info['edge2']
    
    print(f"Computing observed for {metapath}...")
    
    # Load Hetionet edges
    hetionet_edge1 = load_edge_matrix(edge1)
    hetionet_edge2 = load_edge_matrix(edge2)
    
    # Compute metapath
    observed = hetionet_edge1 @ hetionet_edge2
    
    observed_pathways[metapath] = observed
    print(f"  Shape: {observed.shape}, nnz: {observed.nnz:,}")

print(f"\n✓ Observed pathway counts computed for {len(observed_pathways)} metapaths")

## Compute Z-Scores

In [ ]:
print("\nComputing z-scores...\n")

all_z_score_results = []

for mp_info in metapaths_2hop:
    metapath = mp_info['name']
    
    print(f"\n{'='*70}")
    print(f"Processing {metapath}")
    print(f"{'='*70}")
    
    # Load data
    observed = observed_pathways[metapath]
    expected = expected_pathways[metapath]
    variance_matrix, variance_lookup = load_variance_estimates(metapath)
    
    # Convert to dense for calculation
    obs_dense = observed.toarray()
    exp_dense = expected.toarray()
    var_dense = variance_matrix.toarray()
    
    # Compute z-scores: z = (obs - exp) / sqrt(var)
    # Avoid division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        z_scores = (obs_dense - exp_dense) / np.sqrt(var_dense)
        z_scores[~np.isfinite(z_scores)] = 0
    
    # Compute p-values (two-tailed)
    p_values = 2 * (1 - norm.cdf(np.abs(z_scores)))
    
    # Extract non-zero entries
    mask = (obs_dense > min_pathway_count) | (exp_dense > 0)
    sources, targets = np.where(mask)
    
    print(f"\nProcessing {len(sources):,} (source, target) pairs...")
    
    for i, j in zip(sources, targets):
        all_z_score_results.append({
            'metapath': metapath,
            'source_id': int(i),
            'target_id': int(j),
            'observed': float(obs_dense[i, j]),
            'expected': float(exp_dense[i, j]),
            'variance': float(var_dense[i, j]),
            'z_score': float(z_scores[i, j]),
            'p_value': float(p_values[i, j])
        })
    
    print(f"  Mean z-score (non-zero): {z_scores[mask].mean():.4f}")
    print(f"  Std z-score: {z_scores[mask].std():.4f}")
    print(f"  Max |z|: {np.abs(z_scores[mask]).max():.4f}")

print(f"\n{'='*70}")
print(f"✓ Z-scores computed for {len(all_z_score_results):,} total pathway pairs")
print(f"{'='*70}")

## FDR Correction

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(all_z_score_results)

print(f"\nApplying FDR correction (Benjamini-Hochberg)...\n")
print(f"Total tests: {len(results_df):,}")

# Remove infinite/nan p-values
valid_mask = np.isfinite(results_df['p_value']) & (results_df['p_value'] >= 0) & (results_df['p_value'] <= 1)
results_df = results_df[valid_mask].copy()

print(f"Valid tests: {len(results_df):,}")

# Apply FDR correction
reject, p_adj_bh, _, _ = multipletests(results_df['p_value'], method='fdr_bh')
results_df['p_adj_BH'] = p_adj_bh
results_df['significant'] = reject

# Add enrichment/depletion label
results_df['direction'] = results_df['z_score'].apply(
    lambda z: 'enriched' if z > 0 else 'depleted'
)

# Sort by absolute z-score
results_df = results_df.sort_values('z_score', key=lambda x: np.abs(x), ascending=False)

# Summary statistics
n_significant = results_df['significant'].sum()
n_enriched = ((results_df['z_score'] > 0) & results_df['significant']).sum()
n_depleted = ((results_df['z_score'] < 0) & results_df['significant']).sum()

print(f"\nFDR Correction Results (α = {fdr_threshold}):")
print(f"  Significant pairs: {n_significant:,} / {len(results_df):,} ({n_significant/len(results_df)*100:.2f}%)")
print(f"  Enriched: {n_enriched:,}")
print(f"  Depleted: {n_depleted:,}")

# Save all results
results_df.to_csv(results_dir / 'all_z_scores.csv', index=False)
print(f"\n✓ Saved all z-scores: {results_dir / 'all_z_scores.csv'}")

# Save significant results
significant_df = results_df[results_df['significant']].copy()
significant_df.to_csv(results_dir / 'significant_FDR05.csv', index=False)
print(f"✓ Saved significant results: {results_dir / 'significant_FDR05.csv'}")

## Summary by Metapath

In [ ]:
# Summarize by metapath
summary_by_mp = results_df.groupby('metapath').agg({
    'z_score': ['count', 'mean', 'std', lambda x: np.abs(x).max()],
    'significant': 'sum',
    'p_value': lambda x: (x < 0.05).sum()
}).reset_index()

summary_by_mp.columns = ['metapath', 'n_pairs', 'mean_z', 'std_z', 'max_abs_z', 'n_significant_fdr', 'n_significant_nominal']

# Add enriched/depleted counts
enriched_counts = results_df[results_df['significant'] & (results_df['z_score'] > 0)].groupby('metapath').size()
depleted_counts = results_df[results_df['significant'] & (results_df['z_score'] < 0)].groupby('metapath').size()

summary_by_mp = summary_by_mp.merge(enriched_counts.rename('n_enriched'), on='metapath', how='left')
summary_by_mp = summary_by_mp.merge(depleted_counts.rename('n_depleted'), on='metapath', how='left')
summary_by_mp = summary_by_mp.fillna(0)

# Sort by number of significant anomalies
summary_by_mp = summary_by_mp.sort_values('n_significant_fdr', ascending=False)

print("\n" + "="*100)
print("SUMMARY BY METAPATH")
print("="*100)
print(summary_by_mp.to_string(index=False))

# Save summary
summary_by_mp.to_csv(results_dir / 'summary_by_metapath.csv', index=False)
print(f"\n✓ Saved summary: {results_dir / 'summary_by_metapath.csv'}")

## Top Anomalies

In [ ]:
print("\n" + "="*100)
print(f"TOP 20 MOST ANOMALOUS METAPATHS (FDR < {fdr_threshold})")
print("="*100)

top_anomalies = significant_df.head(20)

for idx, row in top_anomalies.iterrows():
    print(f"\n{row['metapath']:12s} | Source {row['source_id']:4d} → Target {row['target_id']:4d}")
    print(f"  z = {row['z_score']:+.2f} | p_adj = {row['p_adj_BH']:.2e} | {row['direction'].upper()}")
    print(f"  Observed: {row['observed']:.1f} | Expected: {row['expected']:.1f} | Variance: {row['variance']:.2f}")

print("\n" + "="*100)

## Visualizations

In [ ]:
# Volcano plot
fig, ax = plt.subplots(figsize=(12, 8))

# Subsample for plotting if too many points
plot_df = results_df.sample(n=min(50000, len(results_df)), random_state=random_seed)

# Plot non-significant
non_sig = plot_df[~plot_df['significant']]
ax.scatter(non_sig['z_score'], -np.log10(non_sig['p_adj_BH']), 
           alpha=0.3, s=10, c='gray', label='Not significant')

# Plot significant
sig = plot_df[plot_df['significant']]
colors = ['red' if z < 0 else 'green' for z in sig['z_score']]
ax.scatter(sig['z_score'], -np.log10(sig['p_adj_BH']), 
           alpha=0.6, s=20, c=colors, label='Significant (FDR < 0.05)')

# Threshold line
ax.axhline(-np.log10(fdr_threshold), color='blue', linestyle='--', linewidth=2, 
           label=f'FDR = {fdr_threshold}')

ax.set_xlabel('Z-score', fontsize=14, fontweight='bold')
ax.set_ylabel('-log₁₀(FDR-adjusted p-value)', fontsize=14, fontweight='bold')
ax.set_title('Volcano Plot: Pathway Anomalies in Hetionet', fontsize=16, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Add text annotations
textstr = f'Total: {len(results_df):,}\n'
textstr += f'Significant: {n_significant:,} ({n_significant/len(results_df)*100:.1f}%)\n'
textstr += f'Enriched: {n_enriched:,}\n'
textstr += f'Depleted: {n_depleted:,}'
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig(results_dir / 'plots' / 'volcano_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved volcano plot")

In [ ]:
# Significant anomalies by metapath
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(summary_by_mp))
width = 0.35

ax.bar(x - width/2, summary_by_mp['n_enriched'], width, label='Enriched', 
       color='green', alpha=0.7)
ax.bar(x + width/2, summary_by_mp['n_depleted'], width, label='Depleted', 
       color='red', alpha=0.7)

ax.set_xlabel('Metapath', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Significant Anomalies', fontsize=12, fontweight='bold')
ax.set_title('Significant Anomalies by Metapath (FDR < 0.05)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(summary_by_mp['metapath'], rotation=45, ha='right')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / 'plots' / 'anomalies_by_metapath.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved metapath summary plot")

In [ ]:
# Z-score distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall distribution
ax = axes[0]
ax.hist(results_df['z_score'], bins=100, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero')
ax.set_xlabel('Z-score', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('Z-score Distribution (All Pathways)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# By metapath
ax = axes[1]
for metapath in results_df['metapath'].unique():
    mp_data = results_df[results_df['metapath'] == metapath]['z_score']
    ax.hist(mp_data, bins=50, alpha=0.5, label=metapath)

ax.axvline(0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Z-score', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('Z-score Distribution by Metapath', fontsize=14, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / 'plots' / 'z_score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved z-score distribution plot")

## Conclusion

In [ ]:
print("\n" + "="*100)
print("ANOMALY DETECTION COMPLETE")
print("="*100)

print(f"\nAnalyzed {len(metapaths_2hop)} metapaths")
print(f"Total pathway pairs: {len(results_df):,}")
print(f"Significant anomalies (FDR < {fdr_threshold}): {n_significant:,} ({n_significant/len(results_df)*100:.2f}%)")
print(f"  Enriched (obs > exp): {n_enriched:,}")
print(f"  Depleted (obs < exp): {n_depleted:,}")

print(f"\nMethod used: {deployment_rec['method']}")
print(f"  N permutations: {deployment_rec['N_required']}")

print(f"\nResults saved to:")
print(f"  - {results_dir / 'all_z_scores.csv'} (all pairs)")
print(f"  - {results_dir / 'significant_FDR05.csv'} (significant only)")
print(f"  - {results_dir / 'summary_by_metapath.csv'} (summary)")
print(f"  - {results_dir / 'plots' / '*.png'} (visualizations)")

print(f"\n{'='*100}")
print(f"PIPELINE COMPLETE!")
print(f"  ✓ Notebook 17: Compositional validation")
print(f"  ✓ Notebook 18: Approximation method comparison")
print(f"  ✓ Notebook 19: Variance estimation")
print(f"  ✓ Notebook 20: Anomaly detection")
print(f"\nBiological insights:")
print(f"  - {n_significant:,} metapaths show significant deviation from degree-based null")
print(f"  - These represent potential biological signals beyond network topology")
print(f"  - See {results_dir / 'significant_FDR05.csv'} for detailed results")
print(f"{'='*100}\n")